# Silver - CFPB Batch Backfill

## Setup

In [1]:
from pyspark.sql.functions import (
    col,
    trim,
    substring,
    to_date,
    when,
    sum as spark_sum,
    current_timestamp,
    lit,
    lower,
    upper
)

batch_id = "cfpb_batch_2026_06_28_to_2026_08_21"

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 3, Finished, Available, Finished, False)

## Read Bronze Table

In [2]:
bronze_df = (
    spark.table("bronze.complaints")
    .filter(col("batch_id") == batch_id)
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 4, Finished, Available, Finished, False)

In [3]:
bronze_df.printSchema()

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 5, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [4]:
display(bronze_df.limit(10))

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 294c4f7d-4f3f-428f-981f-8b87b35cdfd5)

## Remove Duplicate Rows

In [5]:
rows_before_dedup = bronze_df.count()

silver_deduped_df = bronze_df.dropDuplicates()

rows_after_dedup = silver_deduped_df.count()

duplicate_rows_removed = rows_before_dedup - rows_after_dedup
duplicate_rows_removed_pct = (duplicate_rows_removed / rows_before_dedup) * 100

print(f"Rows before deduplication: {rows_before_dedup:,}")
print(f"Rows after deduplication: {rows_after_dedup:,}")
print(f"Rows removed: {duplicate_rows_removed:,}")
print(f"Percentage removed: {duplicate_rows_removed_pct:.6f}%")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 7, Finished, Available, Finished, False)

Rows before deduplication: 1,138,240
Rows after deduplication: 1,138,240
Rows removed: 0
Percentage removed: 0.000000%


## Validate Required Fields

In [6]:
missing_required_fields_condition = (
    col("complaint_id").isNull() |
    (trim(col("complaint_id")) == "") |
    col("date_received").isNull() |
    (trim(col("date_received")) == "") |
    col("product").isNull() |
    (trim(col("product")) == "") |
    col("issue").isNull() |
    (trim(col("issue")) == "") |
    col("company").isNull() |
    (trim(col("company")) == "")
)

missing_required_fields_df = silver_deduped_df.filter(missing_required_fields_condition)
valid_required_fields_df = silver_deduped_df.filter(~missing_required_fields_condition)


StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 8, Finished, Available, Finished, False)

In [7]:
missing_required_fields_count = missing_required_fields_df.count()
valid_required_fields_count = valid_required_fields_df.count()

print(f"Rows missing required fields: {missing_required_fields_count:,}")
print(f"Rows having required fields: {valid_required_fields_count:,}")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 9, Finished, Available, Finished, False)

Rows missing required fields: 0
Rows having required fields: 1,138,240


## Parse Date Fields

In [9]:
silver_dates_df = (
    valid_required_fields_df
    .withColumn(
        "date_received_clean",
        to_date(substring(col("date_received"), 1, 10), "yyyy-MM-dd")
    )
    .withColumn(
        "date_sent_to_company_clean",
        to_date(substring(col("date_sent_to_company"), 1, 10), "yyyy-MM-dd")
    )
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 11, Finished, Available, Finished, False)

In [10]:
display(
    silver_dates_df.select(
        "date_received",
        "date_received_clean",
        "date_sent_to_company",
        "date_sent_to_company_clean"
    ).limit(20)
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8aa6149-ff6d-4806-a7e2-ad1f6484f1c0)

## Validate Date Fields

In [11]:
received_date_missing = (
    col("date_received").isNull()
    | (trim(col("date_received")) == "")
)

received_date_failed_to_parse = (
    col("date_received").isNotNull()
    & (trim(col("date_received")) != "")
    & col("date_received_clean").isNull()
)

sent_date_missing = (
    col("date_sent_to_company").isNull()
    | (trim(col("date_sent_to_company")) == "")
)

sent_date_failed_to_parse = (
    col("date_sent_to_company").isNotNull()
    & (trim(col("date_sent_to_company")) != "")
    & col("date_sent_to_company_clean").isNull()
)

sent_date_before_received_date = (
    col("date_received_clean").isNotNull()
    & col("date_sent_to_company_clean").isNotNull()
    & (col("date_sent_to_company_clean") < col("date_received_clean"))
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 13, Finished, Available, Finished, False)

In [12]:
invalid_date_condition = (
    received_date_missing
    | received_date_failed_to_parse
    | sent_date_failed_to_parse
    | sent_date_before_received_date
)

invalid_date_df = silver_dates_df.filter(invalid_date_condition)

valid_date_df = silver_dates_df.filter(~invalid_date_condition)
     

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 14, Finished, Available, Finished, False)

In [13]:
display(
    silver_dates_df.agg(
        spark_sum(when(received_date_missing, 1).otherwise(0)).alias("missing_date_received"),
        spark_sum(when(received_date_failed_to_parse, 1).otherwise(0)).alias("received_date_failed_to_parse"),
        spark_sum(when(sent_date_missing, 1).otherwise(0)).alias("missing_date_sent_to_company"),
        spark_sum(when(sent_date_failed_to_parse, 1).otherwise(0)).alias("sent_date_failed_to_parse"),
        spark_sum(when(sent_date_before_received_date, 1).otherwise(0)).alias("sent_date_before_received_date")
    )
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, caa224bd-6146-4fe9-a467-ffd8af35cd6b)

In [14]:
missing_sent_date_count = silver_dates_df.filter(sent_date_missing).count()

invalid_date_count = invalid_date_df.count()
valid_date_count = valid_date_df.count()

date_related_findings_count = missing_sent_date_count + invalid_date_count

print(f"Rows with missing date_sent_to_company: {missing_sent_date_count:,}")
print(f"Rows with invalid date logic: {invalid_date_count:,}")
print(f"Total date related findings: {date_related_findings_count:,}")
print(f"Rows with valid dates: {valid_date_count:,}")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 16, Finished, Available, Finished, False)

Rows with missing date_sent_to_company: 0
Rows with invalid date logic: 0
Total date related findings: 0
Rows with valid dates: 1,138,240


## Standardize Location Fields

In [15]:
zip_text = trim(col("zip_code"))

missing_zip = (
    col("zip_code").isNull()
    | (zip_text == "")
    | (lower(zip_text) == "unknown")
)

valid_zip = zip_text.rlike("^[0-9]{5}")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 17, Finished, Available, Finished, False)

In [16]:
silver_location_df = (
    valid_date_df
    .withColumn(
        "state_clean",
        when(
            col("state").isNull() | (trim(col("state")) == ""),
            lit(None)
        ).otherwise(upper(trim(col("state"))))
    )
    .withColumn(
        "zip_code_clean",
        when(missing_zip, lit(None))
        .when(valid_zip, substring(zip_text, 1, 5))
        .otherwise(lit(None))
    )
    .withColumn(
        "zip_code_status",
        when(missing_zip, "missing_zip")
        .when(valid_zip, "valid_zip")
        .otherwise("masked_zip")
    )
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 18, Finished, Available, Finished, False)

In [17]:
display(
    silver_location_df
    .groupBy("zip_code_status")
    .count()
    .orderBy("zip_code_status")
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50635d47-bbf5-418e-a910-e92bf0483796)

## Prepare Final Silver DataFrame

In [18]:
silver_complaints_df = (
    silver_location_df
    .withColumn("silver_processed_timestamp", current_timestamp())
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 20, Finished, Available, Finished, False)

In [19]:
display(silver_complaints_df.limit(10))

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc3267be-f2d9-47fc-b5ca-b16e79e45ea6)

In [20]:
silver_complaints_df.printSchema()

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 22, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- date_received_clean: date (nullable = true)
 |-- date_sent_to_company_clean: date (nullable = true)
 |-- state_clean: string (nullabl

## Prepare Quarantine DataFrame

In [21]:
missing_required_quarantine_df = (
    missing_required_fields_df
    .withColumn("quarantine_reason", lit("missing_required_field"))
)

invalid_date_quarantine_df = (
    invalid_date_df
    .withColumn("quarantine_reason", lit("date_quality_issue"))
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 23, Finished, Available, Finished, False)

In [22]:
quarantine_complaints_df = missing_required_quarantine_df.unionByName(
    invalid_date_quarantine_df,
    allowMissingColumns=True
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 24, Finished, Available, Finished, False)

In [23]:
print(f"Rows in silver_complaints: {silver_complaints_df.count():,}")
print(f"Rows in quarantine_complaints: {quarantine_complaints_df.count():,}")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 25, Finished, Available, Finished, False)

Rows in silver_complaints: 1,138,240
Rows in quarantine_complaints: 0


## Write Silver Tables

In [26]:
silver_complaints_df.createOrReplaceTempView("silver_batch_source")

spark.sql("""
    MERGE INTO silver.complaints AS target
    USING silver_batch_source AS source
        ON target.complaint_id = source.complaint_id

    WHEN NOT MATCHED THEN INSERT *
""")

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 28, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [27]:
(
    quarantine_complaints_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("quarantine.complaints")
)

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 29, Finished, Available, Finished, False)

In [28]:
display(spark.sql("SELECT * FROM silver.complaints LIMIT 10"))

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdc6c557-b7fa-4312-9754-8e32d4b685f1)

In [29]:
display(spark.sql("SELECT * FROM quarantine.complaints LIMIT 10"))

StatementMeta(, a8c5be1a-21c4-480a-ac1f-4821477f6c01, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7644d133-32a0-4fc0-af54-5f032188d684)